In [61]:
import optuna 
import torch
import torch.nn as nn

import gc
import random

import numpy as np
import optuna
import torch
import torch.nn as nn

from torch.utils.data import ConcatDataset, DataLoader, TensorDataset

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()


device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: mps


In [ ]:
import torch
import torch.nn as nn


class DTIRegressor(nn.Module):
    """
    Drug-target affinity regression model.

    Ligand input:
        Morgan fingerprint: [batch_size, fingerprint_size]

    Protein input:
        Integer-encoded amino-acid sequence: [batch_size, sequence_length]

    Output:
        Predicted pKi value: [batch_size]
    """

    def __init__(
        self,
        fingerprint_size=1024,
        protein_vocab_size=22,
        protein_embedding_dim=128,
        padding_idx=0,
        dropout_rate=0.3,
    ):
        super().__init__()

        self.fingerprint_size = fingerprint_size
        self.padding_idx = padding_idx
        self.dropout_rate = dropout_rate


        # -------------------------------------------------
        # Ligand encoder
        # -------------------------------------------------
        # Morgan fingerprint positions do not have a meaningful
        # sequential neighbourhood. Therefore, an MLP is used
        # instead of a one-dimensional CNN.
        self.ligand_encoder = nn.Sequential(
            nn.Linear(fingerprint_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),

            nn.Linear(256, 128),
            nn.ReLU(),
        )

        # -------------------------------------------------
        # Protein embedding
        # -------------------------------------------------
        self.protein_embedding = nn.Embedding(
            num_embeddings=protein_vocab_size,
            embedding_dim=protein_embedding_dim,
            padding_idx=padding_idx,
        )

        # -------------------------------------------------
        # Protein sequence encoder
        # -------------------------------------------------
        # A CNN is appropriate here because amino-acid sequences
        # have meaningful local neighbourhoods and motifs.
        self.protein_encoder = nn.Sequential(
            nn.Conv1d(
                in_channels=protein_embedding_dim,
                out_channels=128,
                kernel_size=5,
                padding=2,
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),

            nn.Conv1d(
                in_channels=128,
                out_channels=128,
                kernel_size=5,
                padding=2,
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )

        # -------------------------------------------------
        # Feature fusion and regression head
        # -------------------------------------------------
        # Ligand representation: 128
        # Protein representation: 128
        # Combined representation: 256
        self.regression_head = nn.Sequential(
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(128, 1),
        )

        self._initialize_weights()

    def _initialize_weights(self):
        """
        Apply suitable initialisation to linear, convolutional
        and embedding layers.
        """
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(
                    module.weight,
                    nonlinearity="relu",
                )

                if module.bias is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.Conv1d):
                nn.init.kaiming_normal_(
                    module.weight,
                    nonlinearity="relu",
                )

                if module.bias is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.Embedding):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=0.02,
                )

                if module.padding_idx is not None:
                    with torch.no_grad():
                        module.weight[module.padding_idx].zero_()

    def masked_max_pool(
        self,
        protein_features,
        protein_mask,
    ):
        """
        Perform max pooling while excluding padded positions.

        Parameters
        ----------
        protein_features:
            Tensor with shape [batch_size, channels, sequence_length].

        protein_mask:
            Boolean tensor with shape [batch_size, sequence_length].
            True indicates a real amino-acid token.

        Returns
        -------
        Tensor with shape [batch_size, channels].
        """

        # Convert mask to [batch_size, 1, sequence_length]
        protein_mask = protein_mask.unsqueeze(1)

        # Replace padded positions with a very small value so
        # they cannot become the maximum.
        protein_features = protein_features.masked_fill(
            ~protein_mask,
            torch.finfo(protein_features.dtype).min,
        )

        pooled_features = protein_features.max(dim=2).values

        # Protect against sequences containing only padding.
        invalid_sequences = ~protein_mask.any(dim=2)

        if invalid_sequences.any():
            pooled_features = pooled_features.masked_fill(
                invalid_sequences,
                0.0,
            )

        return pooled_features

    def forward(self, ligand, protein):
        """
        Forward pass.

        Parameters
        ----------
        ligand:
            Float tensor of shape [batch_size, fingerprint_size].

        protein:
            Long tensor of shape [batch_size, sequence_length].

        Returns
        -------
        Predicted pKi values with shape [batch_size].
        """

        if ligand.ndim != 2:
            raise ValueError(
                "Ligand input must have shape "
                "[batch_size, fingerprint_size]."
            )

        if ligand.size(1) != self.fingerprint_size:
            raise ValueError(
                f"Expected {self.fingerprint_size} ligand features, "
                f"but received {ligand.size(1)}."
            )

        if protein.ndim != 2:
            raise ValueError(
                "Protein input must have shape "
                "[batch_size, sequence_length]."
            )

        if ligand.size(0) != protein.size(0):
            raise ValueError(
                "Ligand and protein inputs must have the same batch size."
            )

        # Ensure correct data types
        ligand = ligand.float()
        protein = protein.long()

        
        # Ligand MLP
        ligand_features = self.ligand_encoder(ligand)

        # Protein CNN
        protein_mask = protein.ne(self.padding_idx)

        # [batch, sequence_length, embedding_dim]
        protein_features = self.protein_embedding(protein)

        # Conv1d expects [batch, channels, sequence_length]
        protein_features = protein_features.permute(0, 2, 1)

        # [batch, 128, sequence_length]
        protein_features = self.protein_encoder(protein_features)

        # [batch, 128]
        protein_features = self.masked_max_pool(
            protein_features,
            protein_mask,
        )

        # Multimodal feature fusion
        combined_features = torch.cat(
            [ligand_features, protein_features],
            dim=1,
        )

        # [batch, 1] -> [batch]
        predictions = self.regression_head(combined_features)

        return predictions.squeeze(-1)

In [63]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler


# -----------------------------
# CONFIG
# -----------------------------
BATCH_SIZE = 16
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(device)

# -----------------------------
# LOAD DATA
# -----------------------------
X_lig = np.load("data/processed_features/X_lig.npy", mmap_mode='r')
X_prot = np.load("data/processed_features/X_prot.npy", mmap_mode='r')
y = np.load("data/processed_features/y.npy", mmap_mode='r')

print("Ligand shape:", X_lig.shape)
print("Protein shape:", X_prot.shape)

# -----------------------------
# SPLIT
# -----------------------------
Xl_train, Xl_test, Xp_train, Xp_test, y_train, y_test = train_test_split(
    X_lig, X_prot, y, test_size=0.2, random_state=42
)

Xl_train, Xl_val, Xp_train, Xp_val, y_train, y_val = train_test_split(
    Xl_train, Xp_train, y_train, test_size=0.2, random_state=42
)

Xl_train
X_prot

scaler = StandardScaler()
y_train_scaled = scaler.fit_transform(y_train.reshape(-1, 1)).flatten()

y_test_scaled = scaler.transform(y_test.reshape(-1, 1)).flatten()

y_val_scaled = scaler.transform(y_val.reshape(-1, 1)).flatten()



mps
Ligand shape: (148111, 1024)
Protein shape: (148111, 320)


In [64]:
class DrugTargetDataset(Dataset):
    def __init__(self, lig, prot, y):
        self.lig = lig
        self.prot = prot
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.lig[idx], dtype=torch.float32),
            torch.tensor(self.prot[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.float32),
        )

In [65]:
train_dataset = DrugTargetDataset(Xl_train, Xp_train, y_train_scaled)
test_dataset = DrugTargetDataset(Xl_test, Xp_test, y_test_scaled)
val_dataset = DrugTargetDataset(Xl_val, Xp_val, y_val_scaled)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [66]:
def train_one_epoch(
    model,
    data_loader,
    optimizer,
    criterion,
    device
):
    model.train()

    total_loss = 0.0
    number_of_samples = 0

    for ligand, protein, labels in data_loader:

        ligand = ligand.to(device)
        protein = protein.to(device)
        labels = labels.to(device)

        # Remove gradients from the previous batch
        optimizer.zero_grad()

        # Forward propagation
        predictions = model(
            ligand,
            protein
        )

        # Calculate MSE loss
        loss = criterion(
            predictions,
            labels
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                "Non-finite training loss detected."
            )

        # Backpropagation
        loss.backward()

        # Reduce the risk of exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        # Update model parameters
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        number_of_samples += batch_size

    average_loss = (
        total_loss / number_of_samples
    )

    return average_loss

In [67]:
def calculate_rmse(
    model,
    data_loader,
    device
):
    model.eval()

    squared_error_sum = 0.0
    number_of_predictions = 0

    with torch.no_grad():

        for ligand, protein, labels in data_loader:

            ligand = ligand.to(device)
            protein = protein.to(device)
            labels = labels.to(device)

            predictions = model(
                ligand,
                protein
            )

            if not torch.isfinite(predictions).all():
                return float("inf")

            squared_errors = (
                predictions - labels
            ) ** 2

            squared_error_sum += (
                squared_errors.sum().item()
            )

            number_of_predictions += labels.numel()

    mse = (
        squared_error_sum /
        number_of_predictions
    )

    rmse = np.sqrt(mse)

    return float(rmse)

In [ ]:
def objective(trial):

    # Use a slightly different seed for each trial
    set_seed(SEED + trial.number)

    # --------------------------------------
    # Hyperparameters selected by Optuna
    # --------------------------------------
    learning_rate = trial.suggest_float("learning_rate",1e-5,1e-3,log=True)
    weight_decay = trial.suggest_float("weight_decay",1e-7,1e-3,log=True)
    batch_size = trial.suggest_categorical("batch_size",[32, 64, 128])
    ligand_dropout = trial.suggest_float("ligand_dropout",0.1,0.5,step=0.1)
    hover_loss_delta = trial.suggest_float("huber_loss_delta",0.5,2.0,step=0.1)
    

    # --------------------------------------
    # DataLoaders for this trial
    # --------------------------------------
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,

        # Prevent a final batch containing one sample.
        # BatchNorm cannot train with batch size 1.
        drop_last=True,
        num_workers=0
    )

    validation_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,

        shuffle=False,
        drop_last=False,
        num_workers=0
    )

    # --------------------------------------
    # Create a new model for every trial
    # --------------------------------------
    model = DTIRegressor(
        dropout_rate=ligand_dropout
    ).to(device)

   
    criterion = nn.HuberLoss(delta=hover_loss_delta)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    maximum_epochs = 30
    patience = 5

    best_validation_rmse = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0

    try:

        for epoch in range(maximum_epochs):

            training_loss = train_one_epoch(
                model=model,
                data_loader=train_loader,
                optimizer=optimizer,
                criterion=criterion,
                device=device
            )

            validation_rmse = calculate_rmse(
                model=model,
                data_loader=validation_loader,
                device=device
            )

            print(
                f"Trial {trial.number:02d} | "
                f"Epoch {epoch + 1:02d} | "
                f"Train MSE: {training_loss:.4f} | "
                f"Validation RMSE: {validation_rmse:.4f}"
            )

            if not np.isfinite(validation_rmse):
                raise optuna.TrialPruned(
                    "Non-finite validation RMSE."
                )

            # Send intermediate performance to Optuna
            trial.report(
                validation_rmse,
                step=epoch
            )

            # Optuna may stop an unpromising trial
            if trial.should_prune():
                raise optuna.TrialPruned()

            # Normal early stopping
            if validation_rmse < best_validation_rmse:

                best_validation_rmse = validation_rmse
                best_epoch = epoch
                epochs_without_improvement = 0

            else:
                epochs_without_improvement += 1

            if epochs_without_improvement >= patience:
                print("Early stopping")
                break

        # Store the best epoch for final training
        trial.set_user_attr(
            "best_epoch",
            best_epoch
        )

        return best_validation_rmse

    except FloatingPointError as error:
        raise optuna.TrialPruned(
            str(error)
        )

    finally:
        del model
        del optimizer
        del train_loader
        del validation_loader

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
sampler = optuna.samplers.TPESampler(
    seed=SEED
)

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=5,
    interval_steps=1
)

study = optuna.create_study(
    study_name="dti_regressor_optuna",
    direction="minimize",
    sampler=sampler,
    pruner=pruner,
    storage="sqlite:///dti_regressor_optuna.db",
    load_if_exists=True
)

study.optimize(
    objective,
    n_trials=30,
    
    # Use one trial at a time on MPS or a single GPU
    n_jobs=1,
    gc_after_trial=True
)

[I 2026-08-06 12:14:15,354] Using an existing study with name 'dti_regressor_optuna' instead of creating a new one.


Trial 20 | Epoch 01 | Train MSE: 0.5203 | Validation RMSE: 0.9076
Trial 20 | Epoch 02 | Train MSE: 0.4054 | Validation RMSE: 0.8732
Trial 20 | Epoch 03 | Train MSE: 0.3674 | Validation RMSE: 0.8549
Trial 20 | Epoch 04 | Train MSE: 0.3438 | Validation RMSE: 0.8397
Trial 20 | Epoch 05 | Train MSE: 0.3234 | Validation RMSE: 0.8292
Trial 20 | Epoch 06 | Train MSE: 0.3037 | Validation RMSE: 0.8222
Trial 20 | Epoch 07 | Train MSE: 0.2879 | Validation RMSE: 0.8150
Trial 20 | Epoch 08 | Train MSE: 0.2730 | Validation RMSE: 0.8180
Trial 20 | Epoch 09 | Train MSE: 0.2592 | Validation RMSE: 0.8109
Trial 20 | Epoch 10 | Train MSE: 0.2485 | Validation RMSE: 0.8063
Trial 20 | Epoch 11 | Train MSE: 0.2372 | Validation RMSE: 0.8099
Trial 20 | Epoch 12 | Train MSE: 0.2266 | Validation RMSE: 0.8069
Trial 20 | Epoch 13 | Train MSE: 0.2187 | Validation RMSE: 0.8142
Trial 20 | Epoch 14 | Train MSE: 0.2103 | Validation RMSE: 0.8081
Trial 20 | Epoch 15 | Train MSE: 0.2038 | Validation RMSE: 0.8124
Early stop

[I 2026-08-06 12:35:12,439] Trial 20 finished with value: 0.8062792483923071 and parameters: {'learning_rate': 5.6115164153345e-05, 'weight_decay': 0.0006351221010640692, 'batch_size': 32, 'ligand_dropout': 0.1, 'protein_dropout': 0.1, 'head_dropout_1': 0.5, 'head_dropout_2': 0.4, 'huber_loss_delta': 1.6}. Best is trial 20 with value: 0.8062792483923071.


Trial 21 | Epoch 01 | Train MSE: 0.5685 | Validation RMSE: 1.0665
Trial 21 | Epoch 02 | Train MSE: 0.5031 | Validation RMSE: 0.9882
Trial 21 | Epoch 03 | Train MSE: 0.4482 | Validation RMSE: 0.9710
Trial 21 | Epoch 04 | Train MSE: 0.4155 | Validation RMSE: 0.9376
Trial 21 | Epoch 05 | Train MSE: 0.3945 | Validation RMSE: 0.9184
Trial 21 | Epoch 06 | Train MSE: 0.3803 | Validation RMSE: 0.9149
Trial 21 | Epoch 07 | Train MSE: 0.3665 | Validation RMSE: 0.9024
Trial 21 | Epoch 08 | Train MSE: 0.3582 | Validation RMSE: 0.8916
Trial 21 | Epoch 09 | Train MSE: 0.3508 | Validation RMSE: 0.8872
Trial 21 | Epoch 10 | Train MSE: 0.3439 | Validation RMSE: 0.8806
Trial 21 | Epoch 11 | Train MSE: 0.3377 | Validation RMSE: 0.8739
Trial 21 | Epoch 12 | Train MSE: 0.3317 | Validation RMSE: 0.8730
Trial 21 | Epoch 13 | Train MSE: 0.3282 | Validation RMSE: 0.8703
Trial 21 | Epoch 14 | Train MSE: 0.3222 | Validation RMSE: 0.8629
Trial 21 | Epoch 15 | Train MSE: 0.3187 | Validation RMSE: 0.8602
Trial 21 |

[I 2026-08-06 13:16:38,088] Trial 21 finished with value: 0.8233819223973304 and parameters: {'learning_rate': 1.0994335574766187e-05, 'weight_decay': 0.0007579479953347995, 'batch_size': 32, 'ligand_dropout': 0.1, 'protein_dropout': 0.2, 'head_dropout_1': 0.4, 'head_dropout_2': 0.30000000000000004, 'huber_loss_delta': 0.9}. Best is trial 20 with value: 0.8062792483923071.


...

Trial 23 | Epoch 01 | Train MSE: 0.7242 | Validation RMSE: 0.9968
Trial 23 | Epoch 02 | Train MSE: 0.4555 | Validation RMSE: 0.9532
Trial 23 | Epoch 03 | Train MSE: 0.4042 | Validation RMSE: 0.9276
Trial 23 | Epoch 04 | Train MSE: 0.3799 | Validation RMSE: 0.9064
Trial 23 | Epoch 05 | Train MSE: 0.3636 | Validation RMSE: 0.9021
Trial 23 | Epoch 06 | Train MSE: 0.3529 | Validation RMSE: 0.9019
Trial 23 | Epoch 07 | Train MSE: 0.3417 | Validation RMSE: 0.9011
Trial 23 | Epoch 08 | Train MSE: 0.3319 | Validation RMSE: 0.8865
Trial 23 | Epoch 09 | Train MSE: 0.3236 | Validation RMSE: 0.8852
Trial 23 | Epoch 10 | Train MSE: 0.3154 | Validation RMSE: 0.8744
Trial 23 | Epoch 11 | Train MSE: 0.3073 | Validation RMSE: 0.8782
Trial 23 | Epoch 12 | Train MSE: 0.2987 | Validation RMSE: 0.8845
Trial 23 | Epoch 13 | Train MSE: 0.2932 | Validation RMSE: 0.8614
Trial 23 | Epoch 14 | Train MSE: 0.2860 | Validation RMSE: 0.8738
Trial 23 | Epoch 15 | Train MSE: 0.2815 | Validation RMSE: 0.8698
Trial 23 |

[I 2026-08-06 13:51:06,481] Trial 23 finished with value: 0.8614242923655134 and parameters: {'learning_rate': 0.000164092867306479, 'weight_decay': 4.809461967501569e-07, 'batch_size': 128, 'ligand_dropout': 0.5, 'protein_dropout': 0.2, 'head_dropout_1': 0.2, 'head_dropout_2': 0.4, 'huber_loss_delta': 1.2000000000000002}. Best is trial 20 with value: 0.8062792483923071.


Trial 23 | Epoch 18 | Train MSE: 0.2647 | Validation RMSE: 0.8850
Early stopping
Trial 24 | Epoch 01 | Train MSE: 0.7140 | Validation RMSE: 1.0362
Trial 24 | Epoch 02 | Train MSE: 0.6154 | Validation RMSE: 1.0035
Trial 24 | Epoch 03 | Train MSE: 0.5022 | Validation RMSE: 0.9991
Trial 24 | Epoch 04 | Train MSE: 0.4309 | Validation RMSE: 0.9896
Trial 24 | Epoch 05 | Train MSE: 0.3975 | Validation RMSE: 0.9888
Trial 24 | Epoch 06 | Train MSE: 0.3760 | Validation RMSE: 0.9798
Trial 24 | Epoch 07 | Train MSE: 0.3589 | Validation RMSE: 0.9576
Trial 24 | Epoch 08 | Train MSE: 0.3460 | Validation RMSE: 0.9414
Trial 24 | Epoch 09 | Train MSE: 0.3371 | Validation RMSE: 0.9310
Trial 24 | Epoch 10 | Train MSE: 0.3305 | Validation RMSE: 0.9266
Trial 24 | Epoch 11 | Train MSE: 0.3246 | Validation RMSE: 0.9243
Trial 24 | Epoch 12 | Train MSE: 0.3200 | Validation RMSE: 0.9199
Trial 24 | Epoch 13 | Train MSE: 0.3175 | Validation RMSE: 0.9194
Trial 24 | Epoch 14 | Train MSE: 0.3139 | Validation RMSE: 0.

[I 2026-08-06 14:19:04,379] Trial 24 finished with value: 0.8791104144196501 and parameters: {'learning_rate': 1.7541893487450798e-05, 'weight_decay': 9.565499215943809e-06, 'batch_size': 64, 'ligand_dropout': 0.4, 'protein_dropout': 0.2, 'head_dropout_1': 0.4, 'head_dropout_2': 0.30000000000000004, 'huber_loss_delta': 0.7}. Best is trial 20 with value: 0.8062792483923071.


Trial 24 | Epoch 30 | Train MSE: 0.2812 | Validation RMSE: 0.8829
Trial 25 | Epoch 01 | Train MSE: 0.3952 | Validation RMSE: 0.8808
Trial 25 | Epoch 02 | Train MSE: 0.3460 | Validation RMSE: 0.8650
Trial 25 | Epoch 03 | Train MSE: 0.3278 | Validation RMSE: 0.8460
Trial 25 | Epoch 04 | Train MSE: 0.3111 | Validation RMSE: 0.8293
Trial 25 | Epoch 05 | Train MSE: 0.2972 | Validation RMSE: 0.8141
Trial 25 | Epoch 06 | Train MSE: 0.2854 | Validation RMSE: 0.8099
Trial 25 | Epoch 07 | Train MSE: 0.2736 | Validation RMSE: 0.7964
Trial 25 | Epoch 08 | Train MSE: 0.2634 | Validation RMSE: 0.7937
Trial 25 | Epoch 09 | Train MSE: 0.2547 | Validation RMSE: 0.8062
Trial 25 | Epoch 10 | Train MSE: 0.2473 | Validation RMSE: 0.7896
Trial 25 | Epoch 11 | Train MSE: 0.2405 | Validation RMSE: 0.7889
Trial 25 | Epoch 12 | Train MSE: 0.2343 | Validation RMSE: 0.7915
Trial 25 | Epoch 13 | Train MSE: 0.2283 | Validation RMSE: 0.7847
Trial 25 | Epoch 14 | Train MSE: 0.2231 | Validation RMSE: 0.7820
Trial 25 |